In [ ]:
!pip install --upgrade transformers datasets faiss-cpu sacrebleu bert-score sentencepiece

In [ ]:
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

dosya_yolu = "/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl"

print("Veri yükleniyor...")
dataset = load_dataset("json", data_files=dosya_yolu)
tum_veri = dataset["train"]

print(f"Toplam satır: {len(tum_veri)}")
print(f"Sütunlar: {tum_veri.column_names}")
print("İlk örnek:", tum_veri[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Veri yükleniyor...
Toplam satır: 48874
Sütunlar: ['input', 'oldf', 'msg', 'y']
İlk örnek: {'input': '@@ -595,8 +595,10 @@ namespace Kratos\n             array_1d<double, 3> b = ZeroVector(3);\n             b[0] = 1.0;\n \n-            const array_1d<double, 3>  c = MathUtils<double>::CrossProduct(a, b);\n-            const array_1d<double, 3>  d = MathUtils<double>::UnitCrossProduct(a, b);\n+            array_1d<double, 3>  c, d;\n+\n+            MathUtils<double>::CrossProduct(c, b, a);\n+            MathUtils<double>::UnitCrossProduct(d, b, a);\n             \n             KRATOS_CHECK_EQUAL(c[2], 2.0);\n             KRATOS_CHECK_EQUAL(d[2], 1.0);', 'oldf': '//    |  /           |\n//    \' /   __| _` | __|  _ \\   __|\n//    . \\  |   (   | |   (   |\\__ `\n//   _|\\_\\_|  \\__,_|\\__|\\___/ ____/\n//                   Multi-Physics\n//\n//  License:\t\t B

In [ ]:
# Sütun adlarını sabitle
INPUT_COL = "input"
LABEL_COL  = "msg"

# Veriyi karıştır ve %90 train / %10 eval olarak böl
split = tum_veri.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
eval_data  = split["test"]

print(f"Train boyutu: {len(train_data)}")
print(f"Eval boyutu:  {len(eval_data)}")

# Küçük deneme seti
DENEME_TRAIN_BOYUTU = 500
DENEME_EVAL_BOYUTU  = 100

kucuk_train = train_data.select(range(DENEME_TRAIN_BOYUTU))
kucuk_eval  = eval_data.select(range(DENEME_EVAL_BOYUTU))

print(f"\nDeneme train: {len(kucuk_train)} örnek")
print(f"Deneme eval:  {len(kucuk_eval)} örnek")

Train boyutu: 43986
Eval boyutu:  4888

Deneme train: 500 örnek
Deneme eval:  100 örnek


In [ ]:
import os
from huggingface_hub import snapshot_download

os.makedirs("./temiz_tokenizer", exist_ok=True)

print("Tokenizer indiriliyor...")
snapshot_download(
    repo_id="Salesforce/codet5p-220m",
    local_dir="./temiz_tokenizer"
)

bozuk_dosyalar = [
    "special_tokens_map.json",
    "tokenizer_config.json",
    "added_tokens.json",
    "tokenizer.json"
]
for dosya in bozuk_dosyalar:
    yol = f"./temiz_tokenizer/{dosya}"
    if os.path.exists(yol):
        os.remove(yol)
        print(f"Silindi: {dosya}")

print("Tokenizer hazır!")

Tokenizer indiriliyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Silindi: special_tokens_map.json
Silindi: tokenizer_config.json
Silindi: added_tokens.json
Tokenizer hazır!


In [ ]:
import torch
import faiss
import numpy as np
import os
from transformers import (
    RobertaTokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

class CodeReviewModelRAG:
    def __init__(self, model_name_or_path="Salesforce/codet5p-220m"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_name = model_name_or_path
        self.index = None
        self.corpus_data = []

        print(f"Cihaz: {self.device.upper()}")
        self.tokenizer = RobertaTokenizer.from_pretrained("./temiz_tokenizer")
        self.model = T5ForConditionalGeneration.from_pretrained(model_name_or_path).to(self.device)
        print(f"Model yüklendi: {model_name_or_path}")

    def _get_embedding(self, text):
        inputs = self.tokenizer(
            text, return_tensors="pt",
            padding=True, truncation=True, max_length=512
        ).to(self.device)
        with torch.no_grad():
            outputs = self.model.encoder(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        return emb

    def build_faiss_index(self, dataset, input_col="input"):
        print(f"FAISS indeksi oluşturuluyor... ({len(dataset)} örnek)")
        embeddings_list = []
        self.corpus_data = []
        for i, item in enumerate(dataset):
            emb = self._get_embedding(item[input_col])
            embeddings_list.append(emb)
            self.corpus_data.append(item)
            if (i + 1) % 200 == 0:
                print(f"  {i+1}/{len(dataset)} işlendi")
        all_embs = np.vstack(embeddings_list).astype("float32")
        self.index = faiss.IndexFlatL2(all_embs.shape[1])
        self.index.add(all_embs)
        print(f"FAISS hazır! {self.index.ntotal} vektör eklendi.")

    def retrieve_similar_examples(self, query_code, k=2):
        if self.index is None:
            return []
        query_emb = self._get_embedding(query_code).astype("float32")
        distances, indices = self.index.search(query_emb, k)
        return [self.corpus_data[idx] for idx in indices[0] if idx != -1]

    def generate_review(self, query_code):
        similar = self.retrieve_similar_examples(query_code, k=2)
        prompt = "Review this code:\n"
        if similar:
            prompt += "Context (Similar Examples):\n"
            for i, ex in enumerate(similar):
                prompt += f"Example {i+1} Review: {ex.get('msg', '')}\n"
        prompt += f"\nCode to review:\n{query_code}\n\nReview:"
        inputs = self.tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=1024
        ).to(self.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=4,
                no_repeat_ngram_size=3,
                early_stopping=True
            )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def train_model(self, train_dataset,
                    output_dir="./results", epochs=3, batch_size=4,
                    input_col="input", label_col="msg"):

        print(f"Eğitim başlıyor... ({len(train_dataset)} train)")
        tokenizer = self.tokenizer

        def preprocess(examples):
            inputs  = ["Review this code: " + str(x) for x in examples[input_col]]
            targets = [str(x) for x in examples[label_col]]
            model_inputs = tokenizer(inputs, max_length=512, truncation=True)
            labels = tokenizer(text_target=targets, max_length=128, truncation=True)
            labels["input_ids"] = [
                [(tok if tok != tokenizer.pad_token_id else -100) for tok in label]
                for label in labels["input_ids"]
            ]
            model_inputs["labels"] = labels["input_ids"]
            return model_inputs

        tokenized_train = train_dataset.map(
            preprocess, batched=True,
            remove_columns=train_dataset.column_names
        )

        training_args = Seq2SeqTrainingArguments(
            output_dir=output_dir,
            eval_strategy="no",
            save_strategy="steps",
            save_steps=500,
            learning_rate=5e-6,
            per_device_train_batch_size=batch_size,
            weight_decay=0.01,
            num_train_epochs=epochs,
            predict_with_generate=False,
            save_total_limit=3,
            load_best_model_at_end=False,
            fp16=False,
            logging_steps=100,
            report_to="none"
        )

        self.trainer = Seq2SeqTrainer(
            model=self.model,
            args=training_args,
            train_dataset=tokenized_train,
            processing_class=self.tokenizer,
            data_collator=DataCollatorForSeq2Seq(
                self.tokenizer, model=self.model, pad_to_multiple_of=8
            ),
        )

        self.trainer.train()

        final_path = os.path.join(output_dir, "final_model")
        self.model.save_pretrained(final_path)
        self.tokenizer.save_pretrained(final_path)
        print(f"\nEğitim tamamlandı! Model kaydedildi: {final_path}")

print("Sınıf tanımlandı ✅")

Sınıf tanımlandı ✅


In [ ]:
rag_system = CodeReviewModelRAG()

rag_system.train_model(
    train_dataset=kucuk_train,
    output_dir="./results_deneme",
    epochs=3,
    batch_size=4,
    input_col=INPUT_COL,
    label_col=LABEL_COL
)

Cihaz: CUDA


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Model yüklendi: Salesforce/codet5p-220m
Eğitim başlıyor... (500 train)


Step,Training Loss
50,7.851953
100,7.227031
150,6.485859
200,6.048828
250,5.969766
300,5.528906
350,5.469297


[transformers] Error during conversion: ReadTimeout('The read operation timed out')
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 77, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. T

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Eğitim tamamlandı! Model kaydedildi: ./results_deneme/final_model


In [ ]:
# Hızlı test — FAISS olmadan direkt model çıktısı
test_kodu = """
def divide(a, b):
    return a / b
"""

inputs = rag_system.tokenizer(
    "Review this code: " + test_kodu,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(rag_system.device)

with torch.no_grad():
    outputs = rag_system.model.generate(
        **inputs,
        max_new_tokens=128,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

sonuc = rag_system.tokenizer.decode(outputs[0], skip_special_tokens=True)
print("--- MODELİN YORUMU ---")
print(sonuc)

--- MODELİN YORUMU ---
Can't be the `


In [ ]:
import os

kayit_yolu = "/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint_deneme"
os.makedirs(kayit_yolu, exist_ok=True)

rag_system.model.save_pretrained(kayit_yolu)
rag_system.tokenizer.save_pretrained(kayit_yolu)

print(f"Deneme modeli Drive'a kaydedildi: {kayit_yolu}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Deneme modeli Drive'a kaydedildi: /content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint_deneme


In [ ]:
DRIVE_KAYIT = "/content/drive/MyDrive/CodeReviewBot/models/tam_egitim"
os.makedirs(DRIVE_KAYIT, exist_ok=True)

print(f"Tam eğitim başlıyor: {len(train_data)} örnek")
print(f"Checkpointlar buraya kaydedilecek: {DRIVE_KAYIT}")

rag_system2 = CodeReviewModelRAG()

rag_system2.train_model(
    train_dataset=train_data,
    output_dir=DRIVE_KAYIT,
    epochs=3,
    batch_size=4,
    input_col=INPUT_COL,
    label_col=LABEL_COL
)

Tam eğitim başlıyor: 43986 örnek
Checkpointlar buraya kaydedilecek: /content/drive/MyDrive/CodeReviewBot/models/tam_egitim
Cihaz: CUDA


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 49, in spawn_con

Model yüklendi: Salesforce/codet5p-220m
Eğitim başlıyor... (43986 train)


Map:   0%|          | 0/43986 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import os
checkpoint_dir = "/content/drive/MyDrive/CodeReviewBot/models/tam_egitim"
checkpoints = [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint-")]
checkpoints.sort(key=lambda x: int(x.split("-")[1]))
print("Kaydedilen checkpointlar:")
for c in checkpoints:
    print(c)

Kaydedilen checkpointlar:
checkpoint-15000
checkpoint-15500
checkpoint-16000


In [ ]:
from transformers import AutoModelForSeq2SeqLM, RobertaTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "/content/drive/MyDrive/CodeReviewBot/models/tam_egitim/checkpoint-15000"
tokenizer = RobertaTokenizer.from_pretrained("./temiz_tokenizer")

# Modeli yükle
print(f"Yükleniyor: {model_path}")
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
model.eval()

# 3 farklı senaryo:
# 1. Hata içeren bir kod (Syntax error veya Mantık)
# 2. Çok basit bir fonksiyon
# 3. Import hatası potansiyeli
test_set = [
    "def divide(a, b): return a / b", # Test 1: Basit
    "def get_user(u): query = 'SELECT * FROM users WHERE name = ' + u; return db.execute(query)", # Test 2: SQL Injection uyarısı bekleriz
    "import os\nprint('hello')" # Test 3: Basit import
]

print("\n--- 15000 Checkpoint Derin Testi ---\n")

for i, code in enumerate(test_set, 1):
    inputs = tokenizer("Review this code: " + code, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, num_beams=4, no_repeat_ngram_size=2)

    print(f"TEST {i}: {code}")
    print(f"Cevap: {tokenizer.decode(outputs[0], skip_special_tokens=True)}\n" + "-"*30)

Yükleniyor: /content/drive/MyDrive/CodeReviewBot/models/tam_egitim/checkpoint-15000


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]


--- 15000 Checkpoint Derin Testi ---

TEST 1: def divide(a, b): return a / b
Cevap: Why do you need this?
------------------------------
TEST 2: def get_user(u): query = 'SELECT * FROM users WHERE name = ' + u; return db.execute(query)
Cevap: Why do you need this?
------------------------------
TEST 3: import os
print('hello')
Cevap: Why do you need this?
------------------------------


In [ ]:
import pandas as pd

# jsonl formatında yükle
df = pd.read_json(
    "/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl",
    lines=True
)

print(f"Toplam satır: {len(df)}")
print(f"Sütunlar: {df.columns.tolist()}")
print()

# En çok tekrarlanan 20 msg
print("=== EN ÇOK TEKRARLANAN 20 YORUM ===")
print(df['msg'].value_counts().head(20))
print()

# msg uzunluk istatistikleri
df['msg_uzunluk'] = df['msg'].str.len()
print("=== MSG UZUNLUK İSTATİSTİKLERİ ===")
print(df['msg_uzunluk'].describe())
print()

# Çok kısa msg'leri gör (5 karakterden az)
kisa = df[df['msg_uzunluk'] < 5]
print(f"5 karakterden kısa msg sayısı: {len(kisa)}")
print(kisa['msg'].value_counts().head(10))

Toplam satır: 48874
Sütunlar: ['input', 'oldf', 'msg', 'y']

=== EN ÇOK TEKRARLANAN 20 YORUM ===
msg
(style) line over 80 characters                                                                        94
(style) trailing whitespace                                                                            35
Prefer single-quoted strings when you don't need string interpolation or special symbols.              32
Prefer double-quoted strings unless you need single quotes to avoid extra backslashes for escaping.    31
Trailing whitespace detected.                                                                          31
Align the parameters of a method call if they span more than one line.                                 25
Please use `String#Equals(String, StringComparison)`                                                   24
Align the elements of a hash literal if they span more than one line.                                  21
Use the new Ruby 1.9 hash syntax.                  

In [ ]:
from transformers import RobertaTokenizer, T5ForConditionalGeneration
import torch

tokenizer = RobertaTokenizer.from_pretrained("./temiz_tokenizer")
model = T5ForConditionalGeneration.from_pretrained(
    "/content/drive/MyDrive/CodeReviewBot/models/tam_egitim/checkpoint-15000"
)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

testler = {
    "Sıfıra bölme riski": "def divide(a, b):\n    return a / b",
    "SQL injection": 'def get_user(u):\n    query = "SELECT * FROM users WHERE name = " + u\n    return db.execute(query)',
    "Sonsuz recursive": "def factorial(n):\n    return n * factorial(n-1)",
    "Temiz kod": "def add(a, b):\n    return a + b",
    "None kontrolü yok": "def get_name(user):\n    return user.name.upper()",
}

for baslik, kod in testler.items():
    inputs = tokenizer(
        "Review this code: " + kod,
        return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    sonuc = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{'='*40}")
    print(f"[{baslik}]")
    print(f"Kod: {kod.strip()}")
    print(f"Model: {sonuc}")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]


[Sıfıra bölme riski]
Kod: def divide(a, b):
    return a / b
Model: Why do you need this?

[SQL injection]
Kod: def get_user(u):
    query = "SELECT * FROM users WHERE name = " + u
    return db.execute(query)
Model: Why do you need this?

[Sonsuz recursive]
Kod: def factorial(n):
    return n * factorial(n-1)
Model: Why do you need this?

[Temiz kod]
Kod: def add(a, b):
    return a + b
Model: Why do you need this?

[None kontrolü yok]
Kod: def get_name(user):
    return user.name.upper()
Model: Why do you need this?
